# Stage 2 campaign: 15 C1 updates from old C1 seed 0 (account ifty1011)

Each update fine-tunes the saved old C1 (seed 0) on singapore_update with one configuration
(data size x seed at 10 epochs, or epochs x seed at full data), evaluates it on singapore_test,
writes its detections, and runs it through the saved C2/C3 of Stage 1b (no retraining) to get
delta1, delta2, delta3 (H1) and the label-free output change (H2). Every model and every result is
saved immediately to /kaggle/working/campaign/.

In [ ]:
import os, sys, glob, json, time, tarfile, hashlib, shutil, subprocess, random
BASE_SEED = 0
from pathlib import Path
T0 = time.time()
def clock(m): print(f'[{time.time()-T0:7.1f}s] {m}', flush=True)

# --- labsd package (from ifty1011/labsd-src) ---
LABSD_DIR = '/kaggle/working/labsd_pkg'; Path(LABSD_DIR).mkdir(parents=True, exist_ok=True)
def locate_labsd():
    for init in glob.glob('/kaggle/input/**/labsd/__init__.py', recursive=True):
        return str(Path(init).parent.parent)
    for tar in glob.glob('/kaggle/input/**/labsd.tar', recursive=True):
        with tarfile.open(tar) as tf: tf.extractall(LABSD_DIR)
        for init in glob.glob(LABSD_DIR + '/**/labsd/__init__.py', recursive=True):
            return str(Path(init).parent.parent)
P = locate_labsd(); assert P, 'labsd not found'
if P not in sys.path: sys.path.insert(0, P)
import labsd
clock('labsd: ' + labsd.__file__)

# --- 350-scene subset (output of ifty1011/labsd-e1-subset350) ---
SPLITS = glob.glob('/kaggle/input/**/splits_350.json', recursive=True)
assert SPLITS, 'splits_350.json not found: attach kernel ifty1011/labsd-e1-subset350'
NUSC_ROOT = str(Path(SPLITS[0]).parent)
assert Path(NUSC_ROOT, 'v1.0-trainval', 'scene.json').exists()
splits = json.load(open(SPLITS[0]))
clock(f'subset root: {NUSC_ROOT}  parts: ' + str({k: len(v) for k, v in splits.items()}))

In [ ]:
clock('installing deps...')
pip = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q']
subprocess.run(pip + ['--no-deps', 'nuscenes-devkit'], check=True)
subprocess.run(pip + ['pyquaternion', 'cachetools', 'descartes', 'shapely'], check=True)
subprocess.run(pip + ['--no-deps', 'ultralytics==8.4.105'], check=True)
subprocess.run(pip + ['ultralytics-thop', 'py-cpuinfo'], check=True)
import torch
assert torch.cuda.is_available(), 'no GPU: enable a GPU accelerator'
cap = torch.cuda.get_device_capability(0)
clock(f'GPU {torch.cuda.get_device_name(0)} sm_{cap[0]}{cap[1]}')
assert cap[0] >= 7, 'GPU below sm_70 (P100): select T4 so training does not fall back to CPU'
# The subset has no maps/ folder, and the devkit checks every map image at start-up.
# Maps are not used here: build a working root that links the subset's metadata and
# images, and points every map record at a 1x1 placeholder image (tested locally).
from PIL import Image
WORK_ROOT = Path('/kaggle/working/nusc_root')
(WORK_ROOT / 'v1.0-trainval').mkdir(parents=True, exist_ok=True)
(WORK_ROOT / 'maps').mkdir(exist_ok=True)
Image.new('L', (1, 1)).save(WORK_ROOT / 'maps' / 'placeholder.png')
if not (WORK_ROOT / 'samples').exists(): os.symlink(Path(NUSC_ROOT) / 'samples', WORK_ROOT / 'samples')
for f in Path(NUSC_ROOT, 'v1.0-trainval').glob('*.json'):
    dst = WORK_ROOT / 'v1.0-trainval' / f.name
    if dst.exists() or dst.is_symlink(): continue
    if f.name == 'map.json':
        recs = json.load(open(f))
        for r in recs: r['filename'] = 'maps/placeholder.png'
        dst.write_text(json.dumps(recs))
    else: os.symlink(f, dst)
from nuscenes.nuscenes import NuScenes
NUSC = NuScenes(version='v1.0-trainval', dataroot=str(WORK_ROOT), verbose=False)
clock(f'NuScenes loaded: {len(NUSC.scene)} scenes in metadata')

In [ ]:
# Campaign inputs (code, test data, saved models). Kaggle may unpack the .tar.
ROOT = Path('/kaggle/working/E1_new'); ROOT.mkdir(parents=True, exist_ok=True)
RUN = glob.glob('/kaggle/input/**/mlpipe/campaign.py', recursive=True)
TAR = glob.glob('/kaggle/input/**/campaign_inputs.tar', recursive=True)
if RUN:
    src = Path(RUN[0]).parent.parent
    for sub in ['mlpipe', 'data', 'models']:
        shutil.copytree(src / sub, ROOT / sub, dirs_exist_ok=True)
elif TAR:
    with tarfile.open(TAR[0]) as tf: tf.extractall(ROOT)
else:
    raise AssertionError('campaign inputs not found: attach dataset <account>/labsd-e1-campaign-inputs')
for p in ['mlpipe/campaign.py', 'data/stage1/singapore_test.json', f'models/stage0/c1_old_s{BASE_SEED}.pt',
          'models/stage1b/p1_isolated_seed0_c2.pt', f'models/stage1b/p1_old_pipeline_c1s{BASE_SEED}_seed0_c3_ego.pt']:
    assert (ROOT / p).exists(), p
st0man = json.load(open(ROOT / 'models/stage0/manifest.json'))
for n, info in st0man['models'].items():
    assert hashlib.sha256(open(ROOT / 'models/stage0' / info['file'], 'rb').read()).hexdigest() == info['sha256'], n
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
clock(f'inputs ready, base old C1 seed {BASE_SEED}')

In [ ]:
from labsd.c1_yolo import build_yolo_dataset, write_yolo_data_yaml, dump_detections, YOLO_CLASS_NAMES
CLS = {c: i for i, c in enumerate(YOLO_CLASS_NAMES)}
DS = '/kaggle/working/ds_sg'
n_tr = build_yolo_dataset(NUSC, splits['singapore_update'], DS, split_name='train')
n_te = build_yolo_dataset(NUSC, splits['singapore_test'], DS, split_name='val')
YAML_FULL = write_yolo_data_yaml(DS)
TRAIN_IMGS = sorted(str(p) for p in Path(DS, 'images', 'train').glob('*.jpg'))
clock(f'singapore_update train images {n_tr} ({len(TRAIN_IMGS)}), test images {n_te}')

def data_yaml(frac, seed):
    """Deterministic image subset per (frac, seed), listed in a text file (no copying)."""
    imgs = list(TRAIN_IMGS); random.Random(seed).shuffle(imgs)
    keep = sorted(imgs[:max(1, round(len(imgs) * frac))])
    txt = Path(DS, f'train_f{int(frac*100)}_s{seed}.txt'); txt.write_text('\n'.join(keep))
    y = Path(DS, f'data_f{int(frac*100)}_s{seed}.yaml')
    y.write_text(f"path: {DS}\ntrain: {txt}\nval: images/val\nnc: {len(YOLO_CLASS_NAMES)}\nnames: {YOLO_CLASS_NAMES}\n")
    return str(y), len(keep)

In [ ]:
from ultralytics import YOLO
from mlpipe import data as MD
from mlpipe.campaign import Evaluator
OUT = Path('/kaggle/working/campaign'); (OUT / 'models').mkdir(parents=True, exist_ok=True)
EXPORT = json.load(open(ROOT / 'data/stage1/singapore_test.json'))
BASE = f'c1_old_s{BASE_SEED}'
BASE_W = str(ROOT / 'models/stage0' / f'{BASE}.pt')
base_mAP50 = st0man['models'][BASE]['test_metrics']['mAP50']

# Same matrix as the earlier 15-update campaign: data size x seed at 10 epochs, epochs x seed at full data
CONFIGS = ([(f, 10, s) for f in (1.0, 0.5, 0.25) for s in (1, 2, 3)] +
           [(1.0, e, s) for e in (5, 20) for s in (1, 2, 3)])
RES_PATH = OUT / f'campaign_base_s{BASE_SEED}.json'
res = json.load(open(RES_PATH)) if RES_PATH.exists() else {'base': BASE, 'base_mAP50': base_mAP50, 'updates': {}}
man_path = OUT / 'models' / 'manifest.json'
man = json.load(open(man_path)) if man_path.exists() else {'base': BASE, 'models': {}}

ev = Evaluator(EXPORT, ROOT / 'models/stage1b', BASE_SEED, log=clock)
res['isolated_test'] = ev.iso
res['seed_noise'] = ev.seed_noise()
json.dump(res, open(RES_PATH, 'w'), indent=1)
clock('evaluator ready; random-variation reference stored')

for frac, ep, seed in CONFIGS:
    tag = f'upd_b{BASE_SEED}_f{int(frac*100)}_e{ep}_s{seed}'
    if tag in res['updates']:
        clock(f'{tag}: already done'); continue
    t = time.time()
    y, n_img = data_yaml(frac, seed)
    w = OUT / 'models' / f'{tag}.pt'
    if not w.exists():
        YOLO(BASE_W).train(data=y, epochs=ep, imgsz=640, batch=8, device='0', seed=seed, val=False,
                           project='/kaggle/working/runs', name=tag, exist_ok=True, verbose=False, plots=False)
        shutil.copy2(f'/kaggle/working/runs/{tag}/weights/last.pt', w)
        shutil.rmtree(f'/kaggle/working/runs/{tag}', ignore_errors=True)
    m = YOLO(str(w)).val(data=YAML_FULL, split='val', imgsz=640, batch=16, device='0', verbose=False, plots=False)
    met = {'mAP50': float(m.box.map50), 'mAP50_95': float(m.box.map), 'precision': float(m.box.mp), 'recall': float(m.box.mr)}
    recs = dump_detections(str(w), splits['singapore_test'], NUSC, score_thresh=0.25)
    by = {}
    for r in recs: by.setdefault(r['sample_token'], []).append([CLS[r['cls']], round(r['x'], 3), round(r['y'], 3), round(r['score'], 4)])
    EXPORT['detections'][tag] = by
    json.dump(by, open(OUT / f'detections_{tag}.json', 'w'))
    man['models'][tag] = {'file': w.name, 'sha256': hashlib.sha256(w.read_bytes()).hexdigest(), 'init': BASE,
                          'frac': frac, 'epochs': ep, 'seed': seed, 'train_images': n_img, 'test_metrics': met,
                          'checkpoint': 'last.pt, val=False'}
    json.dump(man, open(man_path, 'w'), indent=1)
    r = ev.update(tag)
    r.update({'frac': frac, 'epochs': ep, 'seed': seed, 'train_images': n_img, 'test_metrics': met,
              'delta1_mAP50': met['mAP50'] - base_mAP50})
    res['updates'][tag] = r
    json.dump(res, open(RES_PATH, 'w'), indent=1)
    op = r['settings']['old_pipeline']; iso = r['settings']['isolated']
    clock(f"{tag}: d1 {r['delta1_mAP50']:+.4f} | old-pipe d2 {op['delta2_minADE']:+.3f} d3(ego) {op['delta3_L2@3s']['ego']:+.3f} "
          f"d3(noego) {op['delta3_L2@3s']['noego']:+.3f} | iso d2 {iso['delta2_minADE']:+.3f} d3(ego) {iso['delta3_L2@3s']['ego']:+.3f} "
          f"| {time.time()-t:.0f}s")
clock('=== CAMPAIGN DONE ===')